In [58]:
import re
import subprocess
import time
import timeit
from functools import partial
from typing import Callable

import numpy as np
import pandas as pd
import scipy.optimize

from special_quadratic_spline import SpecialQuadraticSpline
from test_special_quadratic_spline import make_random_signal

# Using `timeit`

In [45]:
mean_exec_times = pd.Series(
    index=pd.Index([], name="signal_length"), name="mean_exec_time"
)

for signal_length in range(100, 1000, 100):

    def make_spline() -> SpecialQuadraticSpline:
        return SpecialQuadraticSpline.from_regular_series(
            make_random_signal(signal_length)
        )

    n = 5  # In attempt to reduce noise.
    mean_exec_time = timeit.timeit(make_spline, number=n) / n
    mean_exec_times.loc[signal_length] = mean_exec_time

In [46]:
mean_exec_times.plot(backend="plotly")

Some noise.

# Using `time.process_time`

As mentioned in https://pytest-benchmark.readthedocs.io/en/latest/faq.html.

In [47]:
mean_exec_times = pd.Series(
    index=pd.Index([], name="signal_length"), name="mean_exec_time"
)

for signal_length in range(100, 1000, 100):

    def make_spline() -> SpecialQuadraticSpline:
        return SpecialQuadraticSpline.from_regular_series(
            make_random_signal(signal_length)
        )

    n = 5  # In attempt to reduce noise.
    start_time = time.process_time()
    for _ in range(n):
        make_spline()
    mean_exec_time = (time.process_time() - start_time) / n
    mean_exec_times.loc[signal_length] = mean_exec_time

In [48]:
mean_exec_times.plot(backend="plotly")

Somehow more noise.

# Using `pytest` with the `pytest-benchmark` plugin

In [4]:
pd.DataFrame(
    [
        # From manually running `pytest test_special_quadratic_spline.py`:
        (100, 3.23),
        (101, 3.60),
        (200, 16.04),
        (300, 38.23),
        (400, 72.23),
        (500, 113.59),
        (600, 175.12),
        (700, 262.67),
        (800, 368.17),
        (900, 468.71),
        (1000, 599.74),
    ],
    columns=["signal_length", "mean_exec_time"],
).set_index("signal_length").plot(backend="plotly")

Much better.

## Programmatic use

In [4]:
result = subprocess.run(["pytest", "test_special_quadratic_spline.py"], stdout=subprocess.PIPE)

In [43]:
lines = result.stdout.decode().split("\n")
lines = [re.sub(r"\s+", " ", l) for l in lines]
whitespace_separated_lines = [l.split(" ") for l in lines]
performance_lines = [
    l for l in whitespace_separated_lines if l[0].startswith("test_performance")
]
mean_exec_times = pd.Series(
    {
        int(re.findall(r"test_performance\[(\d+)\]", l[0])[0]): float(
            l[8].replace(",", "")
        )
        for l in performance_lines
    },
    name="mean_exec_time",
).rename_axis("signal_length")

In [110]:
df = mean_exec_times.reset_index()
half_df = df.iloc[: len(df) // 2]  # Analogous to training set.


def zero_intercept_polynomial(x: np.ndarray, *coeffs: list[float]) -> np.ndarray:
    return np.array([coeff * (x ** (i + 1)) for i, coeff in enumerate(coeffs)]).sum(
        axis=0
    )


def get_polynomial_fit(df: pd.DataFrame, deg: int) -> Callable:
    coeffs, _ = scipy.optimize.curve_fit(
        f=zero_intercept_polynomial,
        xdata=df["signal_length"],
        ydata=df["mean_exec_time"],
        p0=np.zeros(deg),
    )
    return lambda x: zero_intercept_polynomial(x, *coeffs)


df["quadratic fit on half_df"] = get_polynomial_fit(half_df, deg=2)(df["signal_length"])
df["cubic fit on half_df"] = get_polynomial_fit(half_df, deg=3)(df["signal_length"])

In [111]:
df.set_index("signal_length").plot(backend="plotly")